# Upload lại Qwen7B adapter đã train — không cần GPU
Bật Internet, thêm/bật Kaggle Secret `HF_TOKEN` có quyền write. Giữ phiên cũ hoặc Add Input Output cũ và sửa `ADAPTER_DIR`.
Cell chỉ dọn snapshot upload tạm của run đã có `completed.json`; giữ nguyên final adapter và checkpoint resume.
Upload chạy trong tiến trình mới, kiểm tra repo private và xác minh file trên commit HF. Không train lại.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys

ADAPTER_DIR = '/kaggle/working/datamate_qwen7b_890/runs/qwen7b_890_fp16_lora/train/final_adapter'
HF_REPO_NAME = 'Zalo-challenge'
# Nếu phiên cũ đã kết thúc: Add Input Output cũ rồi sửa ADAPTER_DIR tới final_adapter đó.
adapter = Path(ADAPTER_DIR).resolve()
for name in ['adapter_config.json', 'adapter_model.safetensors']:
    assert (adapter / name).is_file() and (adapter / name).stat().st_size > 0, f'Thiếu {adapter / name}'

# Only clean temporary upload copies of a completed run; preserve adapter and resume checkpoints.
run_root = adapter.parent.parent
if (run_root / 'train/completed.json').is_file() and adapter.is_relative_to(Path('/kaggle/working').resolve()):
    for name in ['.train-hub-staging', '.smoke-hub-staging']:
        staging = run_root / name
        if staging.is_dir() and not staging.is_symlink():
            target = staging.resolve()
            assert target.parent == run_root and target.name == name
            shutil.rmtree(target)
            print('Đã dọn snapshot upload tạm:', target)
print('Đĩa trống:', round(shutil.disk_usage('/kaggle/working').free / 2**30, 2), 'GiB')
token = os.environ.get('HF_TOKEN')
if not token:
    from kaggle_secrets import UserSecretsClient
    try:
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        raise RuntimeError('Bật Internet và quyền truy cập Kaggle Secret HF_TOKEN có quyền write.') from None
assert token and token.strip(), 'HF_TOKEN rỗng'
environment = dict(os.environ, HF_TOKEN=token.strip(), HF_HUB_OFFLINE='0', TRANSFORMERS_OFFLINE='0')
report = Path('/kaggle/working/hf_recovery_upload.json')
prefix = 'runs/recovered_qwen7b_890/' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')

UPLOAD_SCRIPT = '"""Preflight or retry a private adapter upload in a fresh online process (no GPU)."""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport sys\nimport time\n\ndef safe_error(error):\n    text = str(error)\n    if os.environ.get(\'HF_TOKEN\'):\n        text = text.replace(os.environ[\'HF_TOKEN\'], \'[REDACTED]\')\n    text = re.sub(r\'hf_[A-Za-z0-9_-]+\', \'[REDACTED]\', text)\n    return re.sub(r\'(?i)Bearer\\s+\\S+\', \'Bearer [REDACTED]\', text)[:1500]\n\n\ndef upload(api, repo, prefix, adapter_dir=None):\n    repo_id = repo if \'/\' in repo else api.whoami()[\'name\'] + \'/\' + repo\n    api.create_repo(repo_id=repo_id, repo_type=\'model\', private=True, exist_ok=True)\n    if not api.repo_info(repo_id, repo_type=\'model\').private:\n        raise ValueError(\'Destination repository must be private\')\n    result = dict(repo_id=repo_id, path_in_repo=prefix, status=\'write_verified\')\n    if adapter_dir is None:\n        api.upload_file(repo_id=repo_id, repo_type=\'model\',\n                        path_or_fileobj=b\'{"status":"write_verified"}\\n\',\n                        path_in_repo=prefix + \'/upload_probe.json\', commit_message=\'Verify training backup write access\')\n    else:\n        adapter = Path(adapter_dir).resolve()\n        required = [\'adapter_config.json\', \'adapter_model.safetensors\']\n        for name in required:\n            if not (adapter / name).is_file() or (adapter / name).stat().st_size == 0:\n                raise FileNotFoundError(f\'Missing adapter file: {adapter / name}\')\n        remote = prefix.rstrip(\'/\') + \'/final_adapter\'\n        for attempt in range(3):\n            try:\n                commit = api.upload_folder(repo_id=repo_id, repo_type=\'model\', folder_path=str(adapter),\n                    path_in_repo=remote, ignore_patterns=[\'README.md\'],\n                    commit_message=f\'Upload verified adapter to {remote}\')\n                break\n            except Exception as error:\n                status = getattr(getattr(error, \'response\', None), \'status_code\', None)\n                if isinstance(error, ValueError) or status in (400, 401, 403, 404) or attempt == 2:\n                    raise\n                print(f\'Upload retry {attempt + 1}: {type(error).__name__}: {safe_error(error)}\', flush=True)\n                time.sleep(2 ** attempt)\n        remote_files = set(api.list_repo_files(repo_id=repo_id, repo_type=\'model\', revision=commit.oid))\n        if not all(remote + \'/\' + name in remote_files for name in required):\n            raise RuntimeError(\'Upload returned without both required files at the committed revision\')\n        result.update(status=\'uploaded\', path_in_repo=remote, commit=commit.oid)\n    result[\'url\'] = \'https://huggingface.co/\' + repo_id + \'/tree/main/\' + result[\'path_in_repo\']\n    return result\n\n\ndef main():\n    # Run this script in a fresh process; Hub settings are cached on first import.\n    os.environ[\'HF_HUB_OFFLINE\'] = \'0\'\n    os.environ[\'TRANSFORMERS_OFFLINE\'] = \'0\'\n    os.environ[\'HF_HUB_DISABLE_PROGRESS_BARS\'] = \'1\'\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--repo\', required=True)\n    parser.add_argument(\'--prefix\', required=True)\n    parser.add_argument(\'--adapter-dir\')\n    parser.add_argument(\'--report\', required=True)\n    args = parser.parse_args()\n    if not os.environ.get(\'HF_TOKEN\', \'\').strip():\n        parser.error(\'HF_TOKEN must be provided through the environment/Kaggle Secrets\')\n    from huggingface_hub import HfApi\n    try:\n        result = upload(HfApi(token=os.environ[\'HF_TOKEN\']), args.repo, args.prefix, args.adapter_dir)\n    except Exception as error:\n        result = dict(status=\'failed\', error_type=type(error).__name__, error=safe_error(error),\n                      http_status=getattr(getattr(error, \'response\', None), \'status_code\', None))\n        status = 1\n    else:\n        status = 0\n    report = Path(args.report)\n    report.parent.mkdir(parents=True, exist_ok=True)\n    report.write_text(json.dumps(result, indent=2, ensure_ascii=False) + \'\\n\', encoding=\'utf-8\')\n    print(json.dumps(result, indent=2, ensure_ascii=False), flush=True)\n    return status\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n'
subprocess.run([sys.executable, '-u', '-c', UPLOAD_SCRIPT, '--repo', HF_REPO_NAME,
                '--prefix', prefix, '--adapter-dir', str(adapter), '--report', str(report)],
               env=environment, check=True)
result = json.loads(report.read_text(encoding='utf-8'))
assert result['status'] == 'uploaded'
print('Adapter đã lên HF:', result['url'])
